<a href="https://colab.research.google.com/github/tiagosoaresmm/projeto-dashboard-b2b/blob/main/pedidos_b2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
from datetime import datetime
from google.colab import drive
from google.colab import auth
from google.auth import default
import gspread
import os # Importar o módulo os para listar arquivos

# ==========================================
# 1. AUTENTICAÇÃO E CONEXÃO (MODO MVP)
# ==========================================
print("🔄 Passo 1: Solicitando acesso ao seu Drive e Sheets...")
drive.mount('/content/drive')
auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Autenticação concluída com sucesso!\n")

🔄 Passo 1: Solicitando acesso ao seu Drive e Sheets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Autenticação concluída com sucesso!



In [19]:
# ==========================================
# 2. CONFIGURAÇÃO DOS CAMINHOS DOS ARQUIVOS
# ==========================================
# Seus caminhos exatos salvos em variáveis de texto (strings)
# Base directory where the CSV files are located
base_dir = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'

# Prefixes for the sales and supply files
prefix_vendas = 'cubo_de_vendas_red sales_red'
prefix_supply = 'pedido_supply pedidos_supply'

# Function to find the latest file matching a prefix
def find_latest_csv(directory, prefix):
    matching_files = []
    for filename in os.listdir(directory):
        if filename.startswith(prefix) and filename.endswith('.csv'):
            matching_files.append(os.path.join(directory, filename))

    if not matching_files:
        return None

    # Sort files by modification time (most recent first)
    matching_files.sort(key=os.path.getmtime, reverse=True)
    return matching_files[0]

# Dynamically find the paths
path_vendas = find_latest_csv(base_dir, prefix_vendas)
path_supply = find_latest_csv(base_dir, prefix_supply)

if not path_vendas:
    raise FileNotFoundError(f"No sales CSV file found starting with '{prefix_vendas}' in '{base_dir}'")
if not path_supply:
    raise FileNotFoundError(f"No supply CSV file found starting with '{prefix_supply}' in '{base_dir}'")

print(f"📦 Arquivo de vendas encontrado: {path_vendas}")
print(f"📦 Arquivo de supply encontrado: {path_supply}")

# Nome exato da Planilha Google que você criou para ser o seu banco de dados do Looker
NOME_DA_PLANILHA_DESTINO = "Base_Consolidada_Pedidos"

📦 Arquivo de vendas encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/cubo_de_vendas_red sales_red 2026-05-21T1737.csv
📦 Arquivo de supply encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/pedido_supply pedidos_supply 2026-05-21T1700.csv


In [20]:
# ==========================================
# 3. LEITURA E HIGIENIZAÇÃO DOS DADOS (ETL)
# ==========================================
print("🔄 Passo 2: Carregando os arquivos CSV...")
df_vendas = pd.read_csv(path_vendas)
df_supply = pd.read_csv(path_supply)

print("🔄 Passo 3: Limpando as chaves de cruzamento (Regex)...")
# Garante que o ID de vendas seja string
df_vendas['vendas_id_str'] = df_vendas['Vendas ID Pedido'].astype(str)

# Remove letras e prefixos do ID de Supply (ex: 'Z64985914' vira '64985914')
df_supply['pedido_venda_limpo'] = df_supply['Pedidos Supply Pedido Venda'].astype(str).str.replace(r'\D', '', regex=True)

🔄 Passo 2: Carregando os arquivos CSV...
🔄 Passo 3: Limpando as chaves de cruzamento (Regex)...


In [21]:
import pandas as pd
import numpy as np
from datetime import datetime

# ==========================================
# 4. CRUZAMENTO DAS BASES (JOIN)
# ==========================================
print("🔄 Passo 4: Realizando o Join entre Vendas e Supply (Outer Join)...")

df_consolidado = pd.merge(
    df_vendas,
    df_supply,
    left_on='vendas_id_str',
    right_on='pedido_venda_limpo',
    how='outer',
    indicator=True
)

df_consolidado['Join feito com sucesso'] = df_consolidado['_merge'].apply(lambda x: 1 if x == 'both' else 0)
df_consolidado = df_consolidado.drop(columns=['_merge'])

# Limpeza para garantir que o CSV fique limpo
df_consolidado = df_consolidado.replace([np.inf, -np.inf], np.nan)
df_consolidado = df_consolidado.astype(str)
df_consolidado = df_consolidado.replace(['nan', 'NaN', 'NaT', '<NA>', 'None'], '')

# ==========================================
# 4.1 SALVAR JOIN COMO CSV DIRETO NO DRIVE MONTADO
# ==========================================
data_hora_atual = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
nome_arquivo_csv = f"join_tabelas_{data_hora_atual}.csv"

# Caminho da pasta específica no Drive Compartilhado
caminho_diretorio = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'
caminho_completo_salvamento = f"{caminho_diretorio}{nome_arquivo_csv}"

print(f"🔄 Passo 4.1: Gravando o arquivo diretamente no Drive Compartilhado...")

try:
    # O Pandas grava o arquivo direto na pasta mapeada do Drive
    df_consolidado.to_csv(caminho_completo_salvamento, index=False, encoding='utf-8')

    print(f"✅ SUCESSO! Arquivo salvo diretamente na pasta do projeto.")
    print(f"📁 Caminho: {caminho_completo_salvamento}")

except FileNotFoundError:
    print("❌ Erro: O caminho da pasta não foi encontrado. Certifique-se de que rodou o drive.mount('/content/drive') antes deste passo.")
except Exception as e:
    print(f"❌ Erro ao salvar o arquivo no Drive: {e}")

🔄 Passo 4: Realizando o Join entre Vendas e Supply (Outer Join)...
🔄 Passo 4.1: Gravando o arquivo diretamente no Drive Compartilhado...
✅ SUCESSO! Arquivo salvo diretamente na pasta do projeto.
📁 Caminho: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/join_tabelas_2026-05-21_20-40-46.csv


In [2]:
# ==========================================
# 5. APLICAÇÃO DA REGRA DE NEGÓCIO (STATUS CRÍTICO)
# ==========================================
print("🔄 Passo 5: Calculando pedidos em atraso (Regra de Negócio)...")

# Identifica automaticamente o nome das colunas de data (evita KeyError se o nome mudar)
col_limite = 'Data Limite Last Mile Date' if 'Data Limite Last Mile Date' in df_consolidado.columns else 'Pedidos Supply - Datas Limite Data Limite Last Mile Date'
col_entrega = 'Data Entregue Date' if 'Data Entregue Date' in df_consolidado.columns else 'Pedidos Supply - Datas Data Entregue Date'

# Converte a coluna de prazo limite para o formato de data do Python
df_consolidado[col_limite] = pd.to_datetime(df_consolidado[col_limite], errors='coerce')

# Valida o atraso: Se não foi entregue E a data de hoje passou do prazo limite
if col_entrega in df_consolidado.columns:
    df_consolidado[col_entrega] = pd.to_datetime(df_consolidado[col_entrega], errors='coerce')
    condicao_atraso = (df_consolidado[col_entrega].isna()) & (datetime.now() > df_consolidado[col_limite])
else:
    # Caso a coluna de entrega não esteja na base de teste, valida apenas pelo prazo corrido
    condicao_atraso = (datetime.now() > df_consolidado[col_limite])

# Cria a flag final para o Looker Studio (True = Atrasado / False = Em dia)
df_consolidado['status_critico'] = condicao_atraso

print(f"📊 Processamento concluído! Total de linhas geradas: {len(df_consolidado)}")

# ==========================================
# 6. EXPORTAÇÃO PARA O GOOGLE SHEETS
# ==========================================
print(f"🔄 Passo 6: Exportando dados para a planilha '{NOME_DA_PLANILHA_DESTINO}'...")
try:
    # Abre a planilha mestre criada no seu Drive
    planilha = gc.open(NOME_DA_PLANILHA_DESTINO)
    aba_dados = planilha.get_worksheet(0) # Seleciona a primeira aba

    # Limpa dados anteriores para não acumular sujeira
    aba_dados.clear()

    # Converte os formatos de data e valores nulos do Pandas para texto limpo que o Sheets aceita
    dados_para_salvar = [df_consolidado.columns.values.tolist()] + df_consolidado.fillna('').astype(str).values.tolist()

    # Salva tudo de uma vez só
    aba_dados.update(dados_para_salvar)
    print("\n🚀 PROJETO MVP OPERACIONAL! Dados gravados com sucesso no Google Sheets.")
    print("Agora você já pode abrir o Looker Studio e puxar essa planilha como fonte de dados.")

except Exception as e:
    print(f"\n❌ Erro ao salvar no Google Sheets: {e}")
    print("Dica: Certifique-se de que você criou uma planilha com o nome exato de 'Base_Consolidada_Pedidos' na sua conta.")

🔄 Passo 1: Solicitando acesso ao seu Drive e Sheets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Autenticação concluída com sucesso!

📦 Arquivo de vendas encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/cubo_de_vendas_red sales_red 2026-05-21T1315.csv
📦 Arquivo de supply encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/pedido_supply pedidos_supply 2026-05-21T1314.csv
🔄 Passo 2: Carregando os arquivos CSV...


/tmp/ipykernel_1917/3227027277.py:65: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_supply = pd.read_csv(path_supply)


🔄 Passo 3: Limpando as chaves de cruzamento (Regex)...
🔄 Passo 4: Realizando o Join entre Vendas e Supply...
🔄 Passo 5: Calculando pedidos em atraso (Regra de Negócio)...
📊 Processamento concluído! Total de linhas geradas: 23
🔄 Passo 6: Exportando dados para a planilha 'Base_Consolidada_Pedidos'...

🚀 PROJETO MVP OPERACIONAL! Dados gravados com sucesso no Google Sheets.
Agora você já pode abrir o Looker Studio e puxar essa planilha como fonte de dados.
